# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [3]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("titanic_data_updated.csv")
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [7]:
# YOUR CODE HERE
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
imputer = SimpleImputer(strategy = "constant", fill_value = "Missing")
df['Cabin'] = imputer.fit_transform(df[['Cabin']]).flatten()
df['Deck'] = df['Cabin'].str[0]

In [8]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
817,818,no,second,"Mallet, Mr. Albert",male,31.0,1,1,S.C./PARIS 2079,37.0042,Missing,C,3,M
763,764,yes,first,"Carter, Mrs. William Ernest (Lucile Polk)",female,36.0,1,2,113760,120.0000,B96 B98,S,4,B
325,326,yes,first,"Young, Miss. Marie Grice",female,36.0,0,0,PC 17760,135.6333,C32,C,1,C
641,642,yes,first,"Sagesser, Mlle. Emma",female,24.0,0,0,PC 17477,69.3000,B35,C,1,B
87,88,no,third,"Slocovski, Mr. Selman Francis",male,NaN,0,0,SOTON/OQ 392086,8.0500,Missing,S,1,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [10]:
# YOUR CODE HERE
X = df.drop('Survived', axis = 1)
y = df['Survived']

In [11]:
X.shape

(891, 13)

In [12]:
y.shape

(891,)

---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [14]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42, stratify = y)

In [15]:
X_train.shape

(712, 13)

In [16]:
y_train.shape

(712,)

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [18]:
# YOUR CODE HERE
X_train['z_score'] = (X_train['Age'] - X_train['Age'].mean())/X_train['Age'].std()

In [19]:
valid_rows = X_train['z_score'].abs()<3

In [20]:
X_train = X_train[valid_rows.values]
y_train = y_train[valid_rows.values]

In [21]:
q1 = X_train['Fare'].quantile(0.25) 
q3 = X_train['Fare'].quantile(0.75) 
iqr = q3 - q1
minimum = q1 - 1.5*iqr 
maximum = q3 + 1.5*iqr


In [22]:
X_train['Fare'] = X_train['Fare'].clip(lower = minimum, upper = maximum)

In [23]:
outliers = X_train[X_train['Fare'] > maximum]

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [25]:
# YOUR CODE HERE
#for numerical Column
p1 = Pipeline( 
    steps = [ 
        ("imputation", SimpleImputer(strategy = "mean")),
        ("scaling", StandardScaler()),
        
    ]
)
p2 = Pipeline(
    steps = [
        ("imputation", SimpleImputer(strategy = "median")),
        ("scaling", MinMaxScaler()),
        
    ]
)


---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [27]:
# YOUR CODE HERE
p3 = Pipeline(
    steps = [
        ("cat_imputation", SimpleImputer(strategy = "most_frequent")),
        ("One hot encoding", OneHotEncoder(sparse_output= False, drop = "first", handle_unknown= "ignore"))
    ]
)
categories = [["third", "second", "first"]]
p4 = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("ordinal", OrdinalEncoder(categories= categories)),
        ("MinMax", MinMaxScaler())
            ]
)

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ("p1", p1, ['Age']),
        ("p2", p2, ['Fare','Family_Size']),
        ("p3", p3, ['Embarked', 'Sex', 'Deck']),
        ("p4", p4, ["Pclass"])
    ],remainder = "drop"
)

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [30]:
# YOUR CODE HERE
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [32]:
# YOUR CODE HERE
lr_model = Pipeline(
    steps = [ 
        ("Preprocessing", preprocessor),
        ("logistic regression", LogisticRegression(class_weight = "balanced", max_iter = 1000))
    ]
)

lr_model.fit(X_train, y_train)
y_hat = lr_model.predict(X_test)

---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [60]:
# YOUR CODE HERE
from sklearn.metrics import accuracy_score,precision_score,recall_score
accuracy = accuracy_score(y_test,y_hat)
print(f"Accuracy: {accuracy: .2f}")
precision = precision_score(y_test,y_hat)
print(f"Precision: {precision: .2f}")
recall = recall_score(y_test,y_hat)
print(f"Recall:  {recall}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2894401927.py, line 4)